# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [46]:
# Write your code below.
import os
from dotenv import load_dotenv
from glob import glob
import dask.dataframe as dd


In [ ]:
%pip install dask
%pip install dask[dataframe]

In [10]:
import dask.dataframe as dd

In [47]:
load_dotenv()
price_data_path = os.getenv("PRICE_DATA")

In [13]:
price_data_path = os.getenv('PRICE_DATA')
print(price_data_path)


../../05_src/data/prices/


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [64]:
import os
from glob import glob

# Assuming PRICE_DATA is the environment variable containing the path to your data
PRICE_DATA = os.getenv('PRICE_DATA')

# Use glob to find all parquet files in the directory
parquet_files = glob(os.path.join(PRICE_DATA, '*.parquet'))


# Print the files found to verify
print("Parquet files found:", parquet_files)

Parquet files found: []


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Adjusted Close:
    
    - `returns`: (Adj Close / Adj Close_lag) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [65]:
# Write your code below.



import dask.dataframe as dd

def process_ticker(file):
    try:
        df = dd.read_parquet(file)
        
        # Add lags
        df['Close_lag'] = df['Close'].shift(1)
        df['Adj_Close_lag'] = df['Adj_Close'].shift(1)
        
        # Calculate returns
        df['returns'] = (df['Adj_Close'] / df['Adj_Close_lag']) - 1
        
        # Calculate high-low range
        df['hi_lo_range'] = df['High'] - df['Low']
        
        return df
    except FileNotFoundError:
        print(f"File not found: {file}")
        return None

# Process files and filter out None results
processed_dfs = [process_ticker(file) for file in parquet_files]
valid_dfs = [df for df in processed_dfs if df is not None]

# Concatenate the valid dataframes
if valid_dfs:
    dd_feat = dd.concat(valid_dfs)
else:
    print("No valid parquet files were processed.")
    dd_feat = None


No valid parquet files were processed.


+ Convert the Dask data frame to a pandas data frame. 
+ Add a rolling average return calculation with a window of 10 days.
+ *Tip*: Consider using `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.

# Convert to a Pandas DataFrame for rolling operations
df = dd_feat.compute()

# Calculate the 10-day rolling average for returns
df['rolling_avg_return'] = df['returns'].rolling(10).mean()

# Verify the rolling average column
df.head(15)


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)


Yes, in this context, it was necessary to convert to Pandas to perform the 10-day rolling average calculation easily.

If Dask supported in-place rolling window operations as directly as Pandas, it would be more efficient to perform this calculation in Dask. Dask can handle larger-than-memory data, so performing calculations in Dask prevents memory issues with large datasets.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.